In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW

from evaluate import evaluate_bert_bilstm_crf
from utils.bert_bilstm_crf.data_utils import (
    read_conll_2,
    build_vocab,
    build_tag2idx,
    NERDataset,
    build_collate_fn
)
from models.bert_bilstm_crf import BertBiLstmCrfNER

In [2]:
TRAIN_PATH = "../data/matscholar/train.txt"
VALID_PATH = "../data/matscholar/valid.txt"

MODEL_NAME = "bert-base-cased"
MAX_LENGTH = 128
BATCH_SIZE = 8
JOINT_TRAIN_EPOCHS = 5
BILSTM_ONLY_EPOCHS = 30
WORD_EMBEDDING_DIM = 128
LSTM_HIDDEN_SIZE = 256
DROPOUT = 0.25
BERT_LEARNING_RATE = 3e-5
OTHER_LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

TOTAL_EPOCHS = JOINT_TRAIN_EPOCHS + BILSTM_ONLY_EPOCHS
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
train_sentence, train_tags = read_conll_2(TRAIN_PATH)
valid_sentence, valid_tags = read_conll_2(VALID_PATH)

word2idx = build_vocab(train_sentence)
tag2idx, idx2tag = build_tag2idx(train_tags)

In [4]:
train_dataset = NERDataset(train_sentence, train_tags)
valid_dataset = NERDataset(valid_sentence, valid_tags)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

collate_fn = build_collate_fn(
    tokenizer=tokenizer,
    label2id=tag2idx,
    word2idx=word2idx,
    max_length=MAX_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

model = BertBiLstmCrfNER(
    model_name=MODEL_NAME,
    num_labels=len(tag2idx),
    word_vocab_size=len(word2idx),
    word_embedding_dim=WORD_EMBEDDING_DIM,
    lstm_hidden_size=LSTM_HIDDEN_SIZE,
    dropout=DROPOUT,
    word_pad_idx=word2idx["<PAD>"],
    id2label=idx2tag,
    label2id=tag2idx
).to(DEVICE)

def set_bert_trainable(model, trainable):
    """
    控制 BERT 分支是否参与训练。
    """
    for param in model.bert.parameters():
        param.requires_grad = trainable

def build_optimizer_and_scheduler(train_bert, num_epochs):
    """
    按当前训练阶段构建优化器和学习率调度器。

    train_bert=True 时进行联合训练；
    train_bert=False 时冻结 BERT，只训练 BiLSTM、分类层和 CRF。
    """
    param_groups = []

    if train_bert:
        param_groups.append({"params": model.bert.parameters(), "lr": BERT_LEARNING_RATE})

    param_groups.extend([
        {"params": model.word_embeddings.parameters(), "lr": OTHER_LEARNING_RATE},
        {"params": model.bilstm.parameters(), "lr": OTHER_LEARNING_RATE},
        {"params": model.classifier.parameters(), "lr": OTHER_LEARNING_RATE},
        {"params": model.crf.parameters(), "lr": OTHER_LEARNING_RATE}
    ])

    optimizer = AdamW(param_groups, weight_decay=WEIGHT_DECAY)

    total_steps = len(train_loader) * num_epochs
    warmup_steps = int(WARMUP_RATIO * total_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    return optimizer, scheduler

set_bert_trainable(model, True)
optimizer, scheduler = build_optimizer_and_scheduler(
    train_bert=True,
    num_epochs=JOINT_TRAIN_EPOCHS
)
bert_is_trainable = True

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
best_valid_f1 = 0.0

for epoch in range(TOTAL_EPOCHS):
    if epoch == JOINT_TRAIN_EPOCHS and bert_is_trainable:
        set_bert_trainable(model, False)
        optimizer, scheduler = build_optimizer_and_scheduler(
            train_bert=False,
            num_epochs=BILSTM_ONLY_EPOCHS
        )
        bert_is_trainable = False
        print("\n切换到第二阶段：冻结 BERT，仅继续训练 BiLSTM、分类层和 CRF。")

    model.train()

    # 冻结阶段下，让 BERT 保持 eval 模式，避免 dropout 干扰固定特征。
    if not bert_is_trainable:
        model.bert.eval()

    total_loss = 0.0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        word_input_ids = batch["word_input_ids"].to(DEVICE)
        word_attention_mask = batch["word_attention_mask"].to(DEVICE)
        first_subword_positions = batch["first_subword_positions"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        token_type_ids = batch.get("token_type_ids")

        if token_type_ids is not None:
            token_type_ids = token_type_ids.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            word_input_ids=word_input_ids,
            word_attention_mask=word_attention_mask,
            first_subword_positions=first_subword_positions,
            labels=labels,
            token_type_ids=token_type_ids
        )

        loss = outputs["loss"]
        loss.backward()

        # 梯度裁剪，防止训练过程出现梯度爆炸。
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    train_loss = total_loss / len(train_loader)

    valid_loss, valid_precision, valid_recall, valid_f1, _, _ = evaluate_bert_bilstm_crf(
        model=model,
        dataloader=valid_loader,
        id2label=idx2tag,
        device=DEVICE
    )

    stage_name = "阶段1（联合训练）" if bert_is_trainable else "阶段2（冻结 BERT）"

    print(f"\nEpoch {epoch + 1}/{TOTAL_EPOCHS} - {stage_name}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Precision: {valid_precision:.4f}")
    print(f"Valid Recall: {valid_recall:.4f}")
    print(f"Valid F1: {valid_f1:.4f}")

    if valid_f1 > best_valid_f1:
        best_valid_f1 = valid_f1

        torch.save({
            "model": model.state_dict(),
            "word2idx": word2idx,
            "tag2idx": tag2idx,
            "model_name": MODEL_NAME,
            "word_embedding_dim": WORD_EMBEDDING_DIM,
            "lstm_hidden_size": LSTM_HIDDEN_SIZE,
            "dropout": DROPOUT,
            "joint_train_epochs": JOINT_TRAIN_EPOCHS,
            "bilstm_only_epochs": BILSTM_ONLY_EPOCHS,
            "bert_learning_rate": BERT_LEARNING_RATE,
            "other_learning_rate": OTHER_LEARNING_RATE,
            "best_stage": stage_name

        }, "../models/bert_bilstm_crf.pt")

        print("保存当前最优模型")


Epoch 1/35 - 阶段1（联合训练）
Train Loss: 14.6935
Valid Loss: 5.5046
Valid Precision: 0.7722
Valid Recall: 0.8205
Valid F1: 0.7956
保存当前最优模型

Epoch 2/35 - 阶段1（联合训练）
Train Loss: 3.6548
Valid Loss: 4.6505
Valid Precision: 0.8035
Valid Recall: 0.8566
Valid F1: 0.8292
保存当前最优模型

Epoch 3/35 - 阶段1（联合训练）
Train Loss: 1.6799
Valid Loss: 5.1269
Valid Precision: 0.8199
Valid Recall: 0.8705
Valid F1: 0.8444
保存当前最优模型

Epoch 4/35 - 阶段1（联合训练）
Train Loss: 0.8045
Valid Loss: 6.0680
Valid Precision: 0.8095
Valid Recall: 0.8718
Valid F1: 0.8395

Epoch 5/35 - 阶段1（联合训练）
Train Loss: 0.3905
Valid Loss: 6.6500
Valid Precision: 0.8186
Valid Recall: 0.8688
Valid F1: 0.8429

切换到第二阶段：冻结 BERT，仅继续训练 BiLSTM、分类层和 CRF。

Epoch 6/35 - 阶段2（冻结 BERT）
Train Loss: 0.1033
Valid Loss: 7.1736
Valid Precision: 0.8206
Valid Recall: 0.8666
Valid F1: 0.8430

Epoch 7/35 - 阶段2（冻结 BERT）
Train Loss: 0.0947
Valid Loss: 7.8351
Valid Precision: 0.8210
Valid Recall: 0.8592
Valid F1: 0.8397

Epoch 8/35 - 阶段2（冻结 BERT）
Train Loss: 0.0756
Valid Loss: 